In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers
!pip install faiss-gpu-cu12

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
import os 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device for embedding: {device}")
embedding_model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder', device=device)

if device == 'cuda':
     embedding_model.half() # Convert model to half precision for faster GPU inference

# Filter out any empty or invalid chunks from the DataFrame column
valid_chunks = (
    "Title: " + df['title'].astype(str) + 
    ". Content: " + df['text'].astype(str)
).tolist()

with torch.no_grad():
     doc_embeddings = embedding_model.encode(
         valid_chunks,
         batch_size=512,
         convert_to_numpy=True
     )

doc_embeddings = doc_embeddings.astype('float32')

dimension = doc_embeddings.shape[1]

doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

if device == 'cuda' and len(valid_chunks) > 0:
     print("Creating FAISS index on GPU.")
     res = faiss.StandardGpuResources() # Use a single GPU
     cpu_index = faiss.IndexFlatIP(dimension) # Create CPU index first
     gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index) # Move to GPU (ID 0)
     gpu_index.add(doc_embeddings)
     # Convert back to CPU index before saving
     final_index = faiss.index_gpu_to_cpu(gpu_index)

else:
     print("Creating FAISS index on CPU.")
     final_index = faiss.IndexFlatIP(dimension)
     final_index.add(doc_embeddings)

index_filename = "index.faiss"

output_dir = "/kaggle/working/" 
save_path = os.path.join(output_dir, index_filename)

print(f"Saving FAISS index to: {save_path}")
faiss.write_index(final_index, save_path)
print("FAISS index saved successfully!")